In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
import numpy as np

In [ ]:
(x_train_official, y_train_official), (x_test_official, y_test_official) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [ ]:
x_all = x_all.astype('float32') / 255.0

In [ ]:
x_all = np.concatenate([x_train_official, x_test_official])
y_all = np.concatenate([y_train_official, y_test_official])

In [ ]:
x_all = x_all.astype('float32') / 255.0

In [ ]:
num_classes = 10
y_all = tf.keras.utils.to_categorical(y_all, num_classes)

In [ ]:
TOTAL_IMAGES = 60000
TRAIN_SIZE = 48000
VAL_SIZE = 6000

In [ ]:
x_train, y_train = x_all[:TRAIN_SIZE], y_all[:TRAIN_SIZE]
x_val, y_val = x_all[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE], y_all[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]
x_test, y_test = x_all[TRAIN_SIZE + VAL_SIZE:], y_all[TRAIN_SIZE + VAL_SIZE:]

In [ ]:
from tensorflow.keras.applications import EfficientNetB7
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Resizing
from tensorflow.keras.models import Model

In [ ]:
INPUT_SHAPE = (32, 32, 3)
TARGET_SIZE = 48

In [ ]:
input_tensor = Input(shape=INPUT_SHAPE)
resized_input = Resizing(TARGET_SIZE, TARGET_SIZE)(input_tensor)

In [ ]:
base_model_eff = EfficientNetB7(
    weights='imagenet',
    include_top=False,
    input_tensor=resized_input,
    pooling='avg'
)

258076736/258076736 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
x_eff = base_model_eff.output
predictions_eff = Dense(num_classes, activation='softmax')(x_eff)
model_eff = Model(inputs=input_tensor, outputs=predictions_eff)
base_model_eff.trainable = True

In [ ]:
LEARNING_RATE = 1e-4
BATCH_SIZE = 32
EPOCHS = 10

In [ ]:
OPTIMIZER = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
LOSS_FUNCTION = 'categorical_crossentropy'

In [ ]:
model_eff.compile(
    optimizer=OPTIMIZER,
    loss=LOSS_FUNCTION,
    metrics=['accuracy']
)
print("--- EfficientNetB7 Model Summary (for Parameter Count metric) ---")
model_eff.summary()

--- EfficientNetB7 Model Summary (for Parameter Count metric) ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 48, 48, 3) │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 48, 48, 3) │          0 │ resizing[0][0]    │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 48, 48, 3) │          7 │ rescaling[0][0]   │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 48, 48, 3) │          0 │ normalization[0]… │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 49, 49, 3) │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 24, 24,    │      1,728 │ stem_conv_pad[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 24, 24,    │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 24, 24,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 24, 24,    │        576 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 24, 24,    │        256 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 24, 24,    │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 64)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 64)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 16)  │      1,040 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 64)  │      1,088 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 24, 24,    │          0 │ block1a_activati… │
│ (Multiply)          │ 64)               │            │ block1a_se_expan

 Total params: 64,123,297 (244.61 MB)

 Trainable params: 63,812,570 (243.43 MB)

 Non-trainable params: 310,727 (1.19 MB)

In [ ]:
import time
print(f"Starting EfficientNetB7 Training for {EPOCHS} epochs...")

history_eff = model_eff.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_val, y_val),
    verbose=1
)
print("\n--- EfficientNetB7 Model Evaluation ---")
start_time_eff = time.time()
loss_eff, accuracy_eff = model_eff.evaluate(x_test, y_test, verbose=0)
end_time_eff = time.time()
total_inference_time_eff = end_time_eff - start_time_eff
time_per_image_eff = total_inference_time_eff / x_test.shape[0]
print(f"Final Test Accuracy (EfficientNetB7): {accuracy_eff*100:.2f}%")
print(f"Final Test Loss (EfficientNetB7): {loss_eff:.4f}")
print(f"Total Inference Time (6000 images): {total_inference_time_eff:.4f} seconds")
print(f"Inference Time Per Image (EfficientNetB7): {time_per_image_eff*1000:.2f} ms")

Starting EfficientNetB7 Training for 10 epochs...
Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 431s 136ms/step - accuracy: 0.5290 - loss: 1.3385 - val_accuracy: 0.6718 - val_loss: 0.9606
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 174s 116ms/step - accuracy: 0.8299 - loss: 0.5009 - val_accuracy: 0.4178 - val_loss: 1.7263
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 175s 117ms/step - accuracy: 0.8961 - loss: 0.3007 - val_accuracy: 0.6660 - val_loss: 1.1279
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 175s 117ms/step - accuracy: 0.9385 - loss: 0.1781 - val_accuracy: 0.7248 - val_loss: 1.0721
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 175s 117ms/step - accuracy: 0.9590 - loss: 0.1173 - val_accuracy: 0.8432 - val_loss: 0.5770
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 177s 118ms/step - accuracy: 0.9716 - loss: 0.0856 - val_accuracy: 0.8092 - val_loss: 0.7541
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 199s 116ms/step - accuracy: 0.9771 - loss: 0.0663 - val_accuracy: 0.6458 - val_loss: 1.5361
Epoch 8/10
1